# Humano-no-Bucle: Portões Pré-Ação, Classificação de Risco e Registo de Auditoria

O README desta lição apresenta o Humano-no-Bucle com um pequeno excerto que pede ao utilizador para `APROVAR` ou `REJEITAR` depois do agente já ter produzido uma resposta. Esse padrão é um bom ponto de partida, mas as implementações de HITL em produção normalmente precisam de três peças adicionais:

1. Um **portão pré-ação** que é executado **antes** do agente realizar um passo arriscado, para que o custo, irreversibilidade e latência se mantenham sob controlo.
2. **Classificação por níveis de risco**, de modo que ações de baixo risco sejam executadas automaticamente, ações de risco médio sejam aprovadas em lote, e apenas ações de alto risco dependam da intervenção humana.
3. Um **registo de auditoria mais um ciclo de revisão**, para que todas as decisões do portão sejam registadas em JSONL, e uma rejeição volte a solicitar ao agente uma revisão com uma razão estruturada em vez de simplesmente imprimir `Revising...`.

Este notebook constrói cada um destes elementos com base nos mesmos primitivas que `06-system-message-framework.ipynb`. Executa de ponta a ponta em `DEMO_MODE = True` (não é necessária entrada interativa) ou com pedidos reais de `input()` quando `DEMO_MODE = False`. Nota: no DEMO_MODE o reenvio do terceiro objetivo é encenado para que a mecânica do ciclo seja visível de ponta a ponta. A reclassificação real conduzida por revisão requer `DEMO_MODE = False` e um operador.

**Fora do âmbito (tratado em outras lições):** autenticação e controlo de acesso (ameaça 2 do README da Lição 06), middleware para chamadas de ferramentas (análise detalhada da Lição 14 MAF), padrões de debate multi-agente.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## Padrão 1: Porta pré-ação

O trecho HITL do README chama primeiro o agente, depois pede ao utilizador para aprovar o resultado. Esse é um fluxo **pós-ação**. O agente já executou, portanto o custo da chamada LLM já foi pago, e qualquer efeito secundário (email enviado, linha de base de dados escrita, comentário publicado) já aconteceu.

Um fluxo **pré-ação** insere a porta antes de o agente executar o passo arriscado. O agente propõe a ação, a porta decide se executa, e só com aprovação é que o efeito secundário ocorre.

| Aspecto | Aprovação pós-ação (trecho do README) | Porta pré-ação (este notebook) |
|---|---|---|
| Quando é que a aprovação ocorre? | Depois do agente ter produzido o resultado | Antes de qualquer efeito secundário ser executado |
| Custo LLM em rejeição | Já pago | Pago apenas pela proposta, não pela ação |
| Efeitos secundários irreversíveis | Possível (a ação já aconteceu) | Prevenida |
| Clareza da auditoria | A aprovação é uma declaração impressa | A aprovação é um registo JSON com timestamp, ação, razão |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Padrão 2: Classificação de risco

Nem toda ação necessita de aprovação humana. Uma consulta apenas de leitura numa API pública tem riscos diferentes de enviar um email a um cliente. Tratar ambos da mesma forma desperdiça a atenção do operador e atrasa o agente.

Um modelo simples de 3 níveis:

| Nível | Exemplos | Fluxo de aprovação |
|---|---|---|
| `baixo` (apenas leitura) | Pesquisar numa base de conhecimento, procurar opções de voo, buscar uma página web pública | Execução automática, registada para auditoria |
| `médio` (mutação barata) | Armazenar em cache um resultado, incrementar um contador, agendar um lembrete | Execução automática, mas revisão diária em lote |
| `alto` (exposição externa ou irreversível) | Enviar um email, cobrar um cartão, publicar num canal público | Bloqueio até aprovação humana |

Esta é uma classificação. Sistemas em produção frequentemente usam níveis mais granulares (por exemplo, níveis de permissão AWS IAM, níveis de acesso baseados em funções). A versão de 3 níveis abaixo é a menor versão útil para um agente que mistura ações apenas de leitura e com efeitos secundários.

O classificador abaixo usa heurísticas por palavras-chave para que a demonstração se mantenha determinística e económica. Num sistema em produção, substituiria isto por um classificador treinado ou um motor de políticas.

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Padrão 3: Registo de auditoria e ciclo de revisão

Um `print("Response approved.")` não é um registo de auditoria. Para confiança, cada decisão do gate deve ser registada como um evento estruturado que possa ser posteriormente consultado, reproduzido ou anexado a uma revisão de incidente.

Duas partes:

1. **JSONL só anexar.** Uma linha por decisão, com timestamp, ação, nível, decisão, motivo. Fácil de pesquisar, fácil de enviar para um verdadeiro repositório de logs mais tarde.  
2. **Ciclo de revisão na rejeição.** Quando o gate retorna `deny`, o agente reenvia a si próprio o pedido incluindo o motivo da rejeição no contexto, para que a próxima proposta possa evitar o problema.

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Recursos adicionais

Vários outros projetos públicos implementam variações destes padrões HITL. Compare abordagens para encontrar o que se adequa à sua stack:

- **LangChain** wrappers de ferramentas human-in-the-loop ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): wrappers de ferramentas que pausam a execução para input humano.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ reestruturou isto): usa uma função de agente especificamente para representar o humano em conversas multi-agente.
- **Semantic Kernel** filtros de função ([docs](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): middleware que é executado em torno de cada chamada de função, adequado para lógica de gate.

Cada projeto trata os três sub-padrões de forma diferente: LangChain envolve-os como ferramentas, AutoGen usa um papel de agente, Semantic Kernel usa filtros de middleware. Leia uma ou duas implementações completas antes de escolher um design para o seu próprio agente.

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido utilizando o serviço de tradução automática [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, esteja ciente de que traduções automáticas podem conter erros ou imprecisões. O documento original na sua língua nativa deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas resultantes da utilização desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
